<a href="https://colab.research.google.com/github/bmehak/ViterbiHMM/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import math

# State mapping
states = {"S": 0, "E": 1, "SS": 2, "I": 3, "END": 4}
id2state = {v: k for k, v in states.items()}

# Transition probabilities
transition = np.array([
    [0.0, 1.0, 0.0, 0.0, 0.0],
    [0.0, 0.9, 0.1, 0.0, 0.0],
    [0.0, 0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 0.0, 0.9, 0.1],
    [0.0, 0.0, 0.0, 0.0, 0.0]
])

# Emission mapping
nuc_map = {'A':0, 'C':1, 'G':2, 'T':3}

emission = np.array([
    [0,0,0,0],
    [0.25,0.25,0.25,0.25],
    [0.05,0,0.95,0],
    [0.40,0.10,0.10,0.40],
    [0,0,0,0]
])

sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

In [2]:
n_states = len(states)
n_obs = len(sequence)

NEG_INF = float('-inf')

V = np.full((n_states, n_obs), NEG_INF)
trace = np.zeros((n_states, n_obs), dtype=int)

# Initialization
for s in range(n_states):
    trans = transition[states["S"]][s]
    emis = emission[s][nuc_map[sequence[0]]]

    if trans > 0 and emis > 0:
        V[s][0] = math.log(trans) + math.log(emis)

    trace[s][0] = states["S"]

In [3]:
def compute_step(curr_state, obs_idx):
    obs = nuc_map[sequence[obs_idx]]
    emis = emission[curr_state][obs]

    if emis == 0:
        return NEG_INF, 0

    log_emis = math.log(emis)

    best_val = NEG_INF
    best_prev = 0

    for prev in range(n_states):
        prev_val = V[prev][obs_idx-1]
        trans = transition[prev][curr_state]

        if trans > 0 and prev_val != NEG_INF:
            val = prev_val + math.log(trans) + log_emis

            if val > best_val:
                best_val = val
                best_prev = prev

    return best_val, best_prev

In [4]:
for i in range(1, n_obs):
    for s in range(n_states):
        val, prev = compute_step(s, i)
        V[s][i] = val
        trace[s][i] = prev

In [5]:
def backtrack():
    end = states["END"]
    scores = np.full(n_states, NEG_INF)

    for s in range(n_states):
        t = transition[s][end]
        if t > 0 and V[s][-1] != NEG_INF:
            scores[s] = V[s][-1] + math.log(t)

    last = np.argmax(scores)
    best_prob = scores[last]

    path = [last]

    for i in range(n_obs-1, 0, -1):
        path.append(trace[path[-1]][i])

    path.reverse()

    state_seq = "".join(id2state[p] for p in path)

    print("Sequence:", sequence)
    print("Best Path:", state_seq)
    print("Log Prob:", round(best_prob, 5))

    return state_seq, best_prob

backtrack()

Sequence: CTTCATGTGAAAGCAGACGTAAGTCA
Best Path: EEEEEEEEEEEEEEEEEESSIIIIIII
Log Prob: -41.21968


('EEEEEEEEEEEEEEEEEESSIIIIIII', np.float64(-41.21967768602254))